# Avaliação do Modelo — CreditGuard AI

Carrega o modelo salvo em  e avalia seu desempenho no conjunto de teste.
O split é recriado deterministicamente com  a partir do .

In [ ]:
import pandas as pd
import numpy as np
import joblib
import yaml
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

In [2]:
with open("Model/config.yaml", "r") as f:
    model_cfg = yaml.safe_load(f)

with open("DataPipeline/config.yaml", "r") as f:
    pipeline_cfg = yaml.safe_load(f)

model    = joblib.load(model_cfg["artifacts"]["model"])
features = joblib.load(model_cfg["artifacts"]["features"])

print("Modelo carregado:", model_cfg["artifacts"]["model"])
print("Features:", len(features), "colunas")

Modelo carregado: Model/artifacts/xgb_balanced_model.joblib
Features: 119 colunas


In [3]:
df = pd.read_csv(pipeline_cfg["paths"]["abt"])

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

_, X_test, _, y_test = train_test_split(
    X, y,
    test_size=model_cfg["split"]["test_size"],
    random_state=model_cfg["split"]["random_state"],
    stratify=y
)

print("Conjunto de teste:", X_test.shape)

Conjunto de teste: (61503, 119)


## Métricas de Avaliação

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc     = roc_auc_score(y_test, y_proba)
gini        = 2 * roc_auc - 1
ks          = float(np.max(tpr - fpr))

metrics = {
    "Accuracy" : accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall"   : recall_score(y_test, y_pred),
    "F1-Score" : f1_score(y_test, y_pred),
    "ROC-AUC"  : roc_auc,
    "Gini"     : gini,
    "KS"       : ks,
}

print("Modelo: LightGBM Balanced (candidato a producao)")
print()
for k, v in metrics.items():
    print(f"  {k:12}: {v:.4f}")

## Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Adimplente", "Inadimplente"],
    yticklabels=["Adimplente", "Inadimplente"],
    ax=ax
)
ax.set_title("Matriz de Confusao - LightGBM Balanced")
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

## Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr, tpr, label="LightGBM Balanced (AUC = " + f"{roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Aleatorio (AUC = 0.500)")
ax.set_xlabel("Taxa de Falsos Positivos")
ax.set_ylabel("Taxa de Verdadeiros Positivos")
ax.set_title("Curva ROC - CreditGuard AI")
ax.legend()
plt.tight_layout()
plt.show()

## Interpretabilidade — Feature Importance

A análise de importância de features identifica quais variáveis o modelo utilizou com maior ganho de informação em seus splits. Utilizamos o critério **gain** (redução média da impureza ponderada pelo número de exemplos em cada split), que reflete o poder preditivo efetivo de cada variável — diferente de *weight* (frequência de uso) ou *cover* (cobertura de amostras).

No LightGBM, o gain é acessado via `model.booster_.feature_importance(importance_type='gain')`.

In [ ]:
importances = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=model.booster_.feature_name(),
    name="gain"
).reindex(features, fill_value=0.0).sort_values(ascending=False)

top_n = 20
top_feats = importances.head(top_n)

fig, ax = plt.subplots(figsize=(10, 7))
top_feats[::-1].plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title(f"Top {top_n} Features mais Importantes — LightGBM Balanced (gain)")
ax.set_xlabel("Importância (gain médio por split)")
plt.tight_layout()
plt.show()

print(f"\nRanking das {top_n} variáveis mais importantes (gain):")
for rank, (feat, val) in enumerate(top_feats.items(), 1):
    print(f"  {rank:2d}. {feat:<45} {val:>10.2f}")

As top 10 variáveis pelo critério de ganho concentram a maior parte do poder preditivo do modelo. Os scores externos (`EXT_SOURCE_2`, `EXT_SOURCE_3`) lideram com ampla margem — cada corte nesses atributos reduz substancialmente a impureza dos nós. `DAYS_BIRTH` (idade) e `AMT_CREDIT` aparecem logo em seguida, confirmando que perfil etário e tamanho do crédito são determinantes de risco. A presença de `EXT_SOURCE_1_MISSING` entre as top features valida a decisão de criar flags de ausência durante a preparação dos dados: a ausência do score externo é, por si só, um sinal de inadimplência.

## Conclusão

O modelo **LightGBM Balanced** é o candidato atual para produção (artefato: `lgbm_balanced_model.joblib`, versão v2).

### Comparativo de Modelos — Dataset completo (307.511 registros, split 80/20, `random_state=42`)

| Métrica | LightGBM Balanced (v2) | XGBoost Balanced (v1) | Diferença |
|---|---|---|---|
| Accuracy  | **0.7046** | 0.7049 | −0.0003 |
| Precision | **0.1660** | 0.1655 | +0.0005 |
| Recall    | **0.6606** | 0.6568 | **+0.0038** |
| F1-Score  | **0.2653** | 0.2643 | +0.0010 |
| ROC-AUC   | **0.7524** | 0.7509 | **+0.0015** |
| Gini      | **0.5049** | 0.5019 | +0.0030 |
| KS        | **0.3736** | 0.3694 | **+0.0042** |
| Tempo de treino | **2,89 s** | 4,00 s | −28% |

O LightGBM supera o XGBoost em 6 de 7 métricas e é 28% mais rápido no treinamento com os mesmos hiperparâmetros base (`n_estimators=100`, `max_depth=6`, `learning_rate=0.1`, `scale_pos_weight≈11,39`).

O Recall elevado é prioritário: o custo de conceder crédito a um inadimplente supera o custo de negar crédito a um bom pagador.

### Motivação da migração para LightGBM

- **Velocidade:** GOSS (Gradient-based One-Side Sampling) e EFB (Exclusive Feature Bundling) reduzem o custo computacional por iteração — 28% mais rápido que XGBoost nos mesmos dados
- **Crescimento folha-a-folha:** LightGBM cresce pela folha com maior ganho (`num_leaves=63`), convergindo mais eficientemente que o crescimento nível-a-nível do XGBoost para `max_depth=6`
- **Mesma estratégia de desbalanceamento:** `scale_pos_weight` tem semântica idêntica — negativos/positivos ≈ 11,39
- **Qualidade preditiva superior:** ROC-AUC +0.0015, Recall +0.0038, KS +0.0042 — melhorias consistentes em todas as métricas de risco
- **Compatibilidade com sklearn API:** `predict()`, `predict_proba()`, `feature_importances_` — zero alterações no serviço de inferência (`predict.py`)